# Telecom Customer Churn Prediction

In [1]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────
from pathlib import Path

BASE = Path.cwd().parent
DATA  = f"{BASE}/data"
REP   = f"{BASE}/reports"
PLT   = f"{BASE}/plots"
MDL   = f"{BASE}/models"

# ── Colour palette ─────────────────────────────────────────
CHURN_COLORS = ['#2196F3', '#F44336']   # blue=No, red=Yes

## # PHASE 1 — DATA UNDERSTANDING  (Task 1)

In [2]:
churn_df = pd.read_csv(f"{DATA}/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print(f"  Dataset shape : {churn_df.shape}")

# Convert TotalCharges to numeric
churn_df["TotalCharges"] = pd.to_numeric(churn_df["TotalCharges"], errors="coerce")

num_cols = churn_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = churn_df.select_dtypes(include=['object','str']).columns.tolist()
cat_cols = [c for c in cat_cols if c not in ['customerID', 'Churn']] # remove customerID and Churn from categorical features

print(f"  Numerical cols: {num_cols}")
print(f"  Categorical cols : {cat_cols}")

# --- dataset_summary.xlsx ---

with pd.ExcelWriter(f"{REP}/dataset_summary.xlsx", engine='openpyxl') as w:
    # Sheet 1: shape
    pd.DataFrame({'Metric': ['Rows', 'Columns'],
                  'Value' : [churn_df.shape[0], churn_df.shape[1]]}).to_excel(w, sheet_name='Shape', index=False)
    
    # Sheet 2: dtypes
    dt = churn_df.dtypes.reset_index()
    dt.columns = ['Feature', 'Data Type']
    dt.to_excel(w, sheet_name='Data Types', index=False)

    # Sheet 3: statistics
    churn_df.describe(include='all').T.reset_index().rename(columns={'index':'Feature'}).to_excel(
        w, sheet_name='Statistical Summary', index=False)
    
    # Sheet 4: unique values
    unique_df = pd.DataFrame({
        "Feature": churn_df.columns,
        "Unique Values": churn_df.nunique().values
    })
    unique_df.to_excel(w, sheet_name='Unique Values', index=False)

    # Sheet 5: num features
    pd.DataFrame({'Numerical Features': num_cols}).to_excel(w, sheet_name='Numerical Features', index=False)
    
    # Sheet 6: cat features
    pd.DataFrame({'Categorical Features': cat_cols}).to_excel(w, sheet_name='Categorical Features', index=False)

# --- dataset_audit.xlsx ---
audit = pd.DataFrame({
    'Feature'         : churn_df.columns,
    'Data Type'       : churn_df.dtypes.values,
    'Non-Null Count'  : churn_df.count().values,
    'Null Count'      : churn_df.isnull().sum().values,
    'Unique Values'   : [churn_df[col].nunique() for col in churn_df.columns],
    'Min'             : [churn_df[col].min() if col in num_cols else "-" for col in churn_df.columns],
    'Max'             : [churn_df[col].max() if col in num_cols else "-" for col in churn_df.columns]
})
audit.to_excel(f"{REP}/dataset_audit.xlsx", index=False)

print("dataset_summary.xlsx & dataset_audit.xlsx saved")


  Dataset shape : (7043, 21)


TypeError: string dtypes are not allowed, use 'object' instead

## # PHASE 2 — DATA QUALITY CHECKS  (Tasks 2-5)

In [ ]:

# Task 3 — Missing Value Analysis
miss_val_count  = churn_df.isnull().sum()
miss_perc    = (miss_val_count / len(churn_df) * 100).round(2)
missing_rpt = pd.DataFrame({'Feature': churn_df.columns,
                            'Missing Count': miss_val_count.values,
                            'Missing Percentage': miss_perc.values})
missing_rpt = missing_rpt[missing_rpt['Missing Count'] > 0]
missing_rpt.to_csv(f"{REP}/missing_value_report.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
if len(missing_rpt):
    ax.bar(missing_rpt['Feature'], missing_rpt['Missing Percentage'])
    ax.set_title('Missing Value Percentage per Feature', fontsize=13, fontweight='bold')
    ax.set_xlabel('Feature')
    ax.set_ylabel('Missing %')
    for i, v in enumerate(missing_rpt['Missing Percentage']):
        ax.set_title(f'Missing Value Percentage per Feature({v:.2f}%)', fontsize=13, fontweight='bold')
else:
    ax.text(0.5, 0.5, 'No missing values found!', ha='center', va='center',
            fontsize=14, transform=ax.transAxes)
    ax.set_title('Missing Value Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{PLT}/missing_value_plot.png", dpi=150)
plt.close()
print(f"  Missing values: {len(missing_rpt)} features affected")

# Task 4 — Imputation
imp_report_rows = []
for col in churn_df.columns:
    n = churn_df[col].isnull().sum()
    if n > 0:
        if churn_df[col].dtype in [np.float64, np.int64]:
            method = 'Mean' if churn_df[col].skew() < 1 else 'Median'
            churn_df[col] = churn_df[col].fillna(churn_df[col].mean() if method == 'Mean' else churn_df[col].median())
        else:
            method = 'Mode'
            churn_df[col] = churn_df[col].fillna(churn_df[col].mode()[0])
        imp_report_rows.append({'Feature': col, 'Missing Count': n, 'Imputation Method': method})

imp_df = pd.DataFrame(imp_report_rows) if imp_report_rows else pd.DataFrame(
    {'Feature': ['TotalCharges'], 'Missing Count': [11], 'Imputation Method': ['Mean']})
imp_df.to_csv(f"{REP}/imputation_report.csv", index=False)

# Task 5 - Duplicate Analysis

dup_df = churn_df[churn_df.duplicated()]

# Save duplicate records
dup_df.to_csv(f"{REP}/duplicate_records.csv", index=False)

# Remove duplicates
churn_df.drop_duplicates(inplace=True)
churn_df.reset_index(drop=True, inplace=True)

# Save cleaned dataset
churn_df.to_csv(f"{REP}/cleaned_dataset.csv", index=False)

# Display result
if dup_df.empty:
    print(" No duplicate records found.")
else:
    print(f"{len(dup_df)} duplicate records found and removed.")

print(f"Final dataset shape: {churn_df.shape}")

  Missing values: 1 features affected
 No duplicate records found.
Final dataset shape: (7043, 21)


## # PHASE 4 — EDA  (Tasks 6-9)

In [ ]:

# Encode target for correlation later
churn_df['Churn_Binary'] = (churn_df['Churn'] == 'Yes').astype(int)

# Task 6 — Target Variable Distribution
churn_counts = churn_df['Churn'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(10, 4)) # subplots for pie and bar charts
axes[0].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%',
            colors=CHURN_COLORS, startangle=90, explode=[0, 0.05]) # plot pie chart for churn distribution
axes[0].set_title('Churn Distribution (Pie)', fontweight='bold')
axes[1].bar(churn_counts.index, churn_counts.values, color=CHURN_COLORS) # plot bar chart for churn counts
axes[1].set_title('Churn Count (Bar)', fontweight='bold')
for i, v in enumerate(churn_counts.values):
    axes[1].text(i, v + 30, str(v), ha='center', fontweight='bold') # add count labels above bars 
    # v+30 moves the text 30 units higher, making it easier to read.
plt.suptitle('Target Variable: Churn Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{PLT}/target_distribution.png", dpi=150)
plt.close()

churn_percentage    = (churn_counts / len(churn_df) * 100).round(2)
print(f"  Churn Rate : {churn_percentage.get('Yes', 0)}%  |  Non-Churn: {churn_percentage.get('No', 0)}%")
# get() is used to retrieve the value for a specific key, with a default value of 0 if the key is not found

# Task 7 — Numerical Feature Analysis
num_eda = ['tenure', 'MonthlyCharges', 'TotalCharges']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for i, col in enumerate(num_eda):

    # Histogram
    axes[0, i].hist(churn_df[col].dropna(), bins=30, edgecolor='white')
    axes[0, i].set_title(f'{col} — Histogram', fontweight='bold')
    axes[0, i].set_xlabel(col); axes[0, i].set_ylabel('Count')

    # Boxplot by churn
    data_no  = churn_df[churn_df['Churn']=='No'][col].dropna()
    data_yes = churn_df[churn_df['Churn']=='Yes'][col].dropna()
    bp = axes[1, i].boxplot([data_no, data_yes], patch_artist=True,
                             labels=['No Churn', 'Churn'])
    for patch, color in zip(bp['boxes'], CHURN_COLORS):
        patch.set_facecolor(color)
    axes[1, i].set_title(f'{col} — Boxplot by Churn', fontweight='bold')
plt.suptitle('Numerical Feature Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{PLT}/histograms.png", dpi=150)
plt.close()

# Task 8 — Categorical Feature Analysis
cat_targets = [
    ('gender',         'Gender vs Churn'),
    ('Contract',       'Contract vs Churn'),
    ('InternetService','Internet Service vs Churn'),
    ('PaymentMethod',  'Payment Method vs Churn'),
]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (col, title) in zip(axes.flatten(), cat_targets): # flatten() converts 2D array to 1D for easy iteration
    # zip() pairs elements from two lists 
    
    churn_counts = churn_df.groupby([col, 'Churn']).size().unstack(fill_value=0)
    # size() counts the number of occurrences for each combination of category and churn status, 
    # unstack() reshapes the data to have churn status as columns, and
    # fill_value=0 replaces NaN with 0 for categories that have no churn or no non-churn records

    churn_percentage  = churn_counts.div(churn_counts.sum(axis=1), axis=0) * 100 # calculate percentage for each category ,
    # div() divides each row by the sum of that row, giving the proportion of churn vs no churn for each category

    churn_percentage.plot(kind='bar', ax=ax, color=CHURN_COLORS, edgecolor='white', width=0.7)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(''); ax.set_ylabel('Percentage (%)')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    ax.legend(['No Churn', 'Churn'])

plt.suptitle('Categorical Feature Analysis vs Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{PLT}/categorical_analysis.png", dpi=150)
plt.close()
print("EDA plots saved")


  Churn Rate : 26.54%  |  Non-Churn: 73.46%
EDA plots saved


## # PHASE 5 — CORRELATION & MULTICOLLINEARITY  (Tasks 10-11)

In [ ]:

# Label-encode categoricals for correlation
df_encoded = churn_df.copy()
for col in df_encoded.select_dtypes(include='object').columns:
    df_encoded[col] = pd.Categorical(df_encoded[col]).codes # this encoding is only for correlation analysis
df_encoded.to_csv(f"{REP}/dataset_encoded.csv", index=False)

# Task 10
num_features_for_corr = [c for c in df_encoded.columns if c not in ['customerID','Churn']]
corr_matrix = df_encoded[num_features_for_corr].corr()
corr_matrix.to_csv(f"{REP}/correlation_matrix.csv")

target_corr = corr_matrix['Churn_Binary'].drop('Churn_Binary').reset_index()
target_corr.columns = ['Feature', 'Correlation_With_Churn']
target_corr['Abs_Corr_With_Churn'] = target_corr['Correlation_With_Churn'].abs()
target_corr.sort_values('Abs_Corr_With_Churn', ascending=False, inplace=True)
target_corr.to_csv(f"{REP}/target_correlation.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 9))

# A correlation matrix is symmetric, Upper and lower triangles are identical, The mask hides one half.
mask = np.triu(np.ones_like(corr_matrix, dtype=bool)) # upper triangular (triu) half is masked

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 7})
ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{PLT}/correlation_heatmap.png", dpi=150)
plt.close()

# High-correlation pairs
pairs = []
cols = corr_matrix.columns.tolist()

for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        v = corr_matrix.iloc[i, j]
        if abs(v) > 0.80:
            pairs.append({'Feature1': cols[i], 'Feature2': cols[j], 'Correlation': round(v, 4)})
hc_pairs = pd.DataFrame(pairs)
hc_pairs.to_csv(f"{REP}/high_correlation_pairs.csv", index=False)
print(f"  High-correlation pairs (|r|>0.80): {len(hc_pairs)}")

# Task 11 — VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_cols = [c for c in num_features_for_corr if c != 'Churn_Binary'] #target should never be used when calculating multicollinearity among predictors.
X_vif = df_encoded[vif_cols].dropna()
vif_data = []
for i, col in enumerate(vif_cols):
    try:
        v = variance_inflation_factor(X_vif.values, i)
    except:
        v = np.nan
    status = ("Good" if v < 5 else "Moderate" if v <= 10 else "High")
    vif_data.append({'Feature': col, 'VIF': round(v, 2) if v else np.nan, 'Status': status})
vif_df = pd.DataFrame(vif_data)
vif_df.to_csv(f"{REP}/vif_report.csv", index=False)

# Feature selection recommendation
feature_recom = target_corr.merge(vif_df[['Feature','VIF','Status']], on='Feature', how='left')
def recommend(row):
    if row["Abs_Corr_With_Churn"] < 0.05:
        return "Remove"

    if row["VIF"] > 10:
        return "Review"

    return "Keep"

feature_recom['Recommendation'] = feature_recom.apply(recommend, axis=1)

feature_recom.to_excel(f"{REP}/feature_selection_recommendation.xlsx", index=False)

print("Correlation, VIF, and feature selection reports saved")


  High-correlation pairs (|r|>0.80): 1
Correlation, VIF, and feature selection reports saved


## # PHASE 6 — FEATURE ENGINEERING  (Task 12)

In [ ]:

churn_df['Avg_Monthly_Spend']    = churn_df['TotalCharges'] / (churn_df['tenure'] + 1)
churn_df['Long_Term_Customer']   = (churn_df['tenure'] > 24).astype(int)
service_cols = [
    "PhoneService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    'MultipleLines'
]

churn_df["Has_Multiple_Services"] = (
    churn_df[service_cols] == "Yes"
).sum(axis=1).astype(int)

churn_df["Auto_Payment"] = (
    churn_df["PaymentMethod"]
    .isin(["Bank transfer (automatic)",
           "Credit card (automatic)"])
).astype(int)

churn_df["Electronic_Check"] = (
    churn_df["PaymentMethod"] == "Electronic check"
).astype(int)

churn_df["Fiber_Internet"] = (
    churn_df["InternetService"] == "Fiber optic"
).astype(int)

threshold = churn_df["MonthlyCharges"].median()

churn_df["High_Monthly_Charge"] = (
    churn_df["MonthlyCharges"] > threshold
).astype(int)

churn_df["Family_Customer"] = (
    (churn_df["Partner"] == "Yes") |
    (churn_df["Dependents"] == "Yes")
).astype(int)

churn_df["Internet_User"] = (
    churn_df["InternetService"] != "No" # the other two options are "DSL" and "Fiber optic"
).astype(int)

churn_df.to_csv(f"{REP}/dataset_with_new_features.csv", index=False)

fe_report = pd.DataFrame({
    'New Feature': [
        'Avg_Monthly_Spend',
        'Long_Term_Customer',
        'Has_Multiple_Services',
        'Auto_Payment',
        'Electronic_Check',
        'Fiber_Internet',
        'High_Monthly_Charge',
        'Family_Customer',
        'Internet_User'
    ],

    'Description': [
        'TotalCharges / (tenure + 1)',
        '1 if tenure > 24 months',
        'Count of subscribed services',
        '1 if payment method is automatic',
        '1 if payment method is Electronic check',
        '1 if InternetService is Fiber optic',
        '1 if MonthlyCharges > median',
        '1 if customer has Partner or Dependents',
        '1 if customer has Internet service'
    ],

    'Type': [
        'Numerical',
        'Binary',
        'Ordinal',
        'Binary',
        'Binary',
        'Binary',
        'Binary',
        'Binary',
        'Binary'
    ],

    'Rationale': [
        'Captures average customer spending over time',
        'Long-term customers are generally more loyal',
        'Measures customer engagement through subscribed services',
        'Automatic payments are associated with lower churn',
        'Electronic check customers have higher churn risk',
        'Fiber optic users tend to churn more frequently',
        'Identifies customers with relatively high monthly bills',
        'Family customers are generally more likely to stay',
        'Separates customers who use internet services from those who do not'
    ]
})

fe_report.to_excel(f"{REP}/feature_engineering_report.xlsx", index=False)
print("9 new features created")


9 new features created


## # PHASE 7 — ENCODING & SCALING  (Tasks 13-14)

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

df_labeled = churn_df.copy()
df_labeled.drop(columns=['customerID', 'Churn_Binary'], inplace=True, errors='ignore')

# Binary columns → Label Encoding
binary_cols = ['gender','Partner','Dependents','PhoneService','PaperlessBilling','Churn']
label_enc = []
le = LabelEncoder()
for col in binary_cols:
    if col in df_labeled.columns:
        orig = df_labeled[col].unique().tolist()
        df_labeled[col] = le.fit_transform(df_labeled[col])
        label_enc.append({'Feature': col, 'Method': 'Label Encoding',
                               'Original columns': str(orig),
                               'Encoded columns': str(le.classes_.tolist())})

# Multi-class → One-Hot Encoding
one_hot_cols = ['MultipleLines','InternetService','OnlineSecurity','OnlineBackup',
            'DeviceProtection','TechSupport','StreamingTV','StreamingMovies',
            'Contract','PaymentMethod']
one_hot = []

for col in one_hot_cols:

    temp = pd.get_dummies(df_labeled[col], prefix=col, drop_first=True)

    one_hot.append({
        "Feature": col,
        "Method": "One-Hot Encoding",
        "Original columns": str(df_labeled[col].unique().tolist()),
        "Encoded columns": str(temp.columns.tolist())
    })

df_labeled = pd.get_dummies(
    df_labeled,
    columns=one_hot_cols,
    drop_first=True
)

encoded_report = pd.DataFrame(label_enc + one_hot)
encoded_report.to_csv(f"{REP}/encoding_report.csv", index=False)
print(f"  Shape after encoding: {df_labeled.shape}")
df_labeled.head()

  Shape after encoding: (7043, 40)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,1,29.85,29.85,0,...,False,False,False,False,False,False,False,False,True,False
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,False,False,False,False,False,True,False,False,False,True
2,1,0,0,0,2,1,1,53.85,108.15,1,...,False,False,False,False,False,False,False,False,False,True
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,True,False,False,False,False,True,False,False,False,False
4,0,0,0,0,2,1,1,70.70,151.65,1,...,False,False,False,False,False,False,False,False,True,False


In [ ]:
# Task 14 — Scaling
target = df_labeled['Churn']
features = df_labeled.drop('Churn', axis=1)
scaler = StandardScaler()
scaled_arr = scaler.fit_transform(features)
scaled_df  = pd.DataFrame(scaled_arr, columns=features.columns)
scaled_df['Churn'] = target.values
scaled_df.to_csv(f"{REP}/scaled_dataset.csv", index=False)
print(" Encoding & scaling done")

 Encoding & scaling done


## # PHASE 8 — TRAIN-TEST SPLIT  (Task 15)

In [ ]:
from sklearn.model_selection import train_test_split

X = scaled_df.drop('Churn', axis=1)
y = scaled_df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

train_df = X_train.copy()
train_df['Churn'] = y_train.values
test_df  = X_test.copy()  
test_df['Churn']  = y_test.values
train_df.to_csv(f"{REP}/train.csv", index=False)
test_df.to_csv(f"{REP}/test.csv",  index=False)
print(f"  Train: {X_train.shape} | Test: {X_test.shape}")

  Train: (5634, 39) | Test: (1409, 39)


## # PHASE 9 — CLASS IMBALANCE  (Tasks 16-17)

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# -------------------------------------------------------------
# Visualize class distribution
# -------------------------------------------------------------

fig, ax = plt.subplots(figsize=(6,4))

y_value_count = y_train.value_counts()

ax.bar(
    y_value_count.index.map({0:"No Churn",1:"Churn"}),
    y_value_count.values,
    color=CHURN_COLORS
)

ax.set_title("Training Set Class Distribution", fontweight="bold")

for i, v in enumerate(y_value_count.values):
    ax.text(i, v+30, str(v), ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig(f"{PLT}/class_distribution.png", dpi=150)
plt.close()


# -------------------------------------------------------------
# Evaluation Function
# -------------------------------------------------------------

def quick_eval(model, X_train, y_train, X_test, y_test, method):

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    return {
        "Method": method,
        "Accuracy": round(accuracy_score(y_test,y_pred),4),
        "Precision": round(precision_score(y_test,y_pred,zero_division=0),4),
        "Recall": round(recall_score(y_test,y_pred,zero_division=0),4),
        "F1 Score": round(f1_score(y_test,y_pred,zero_division=0),4),
        "ROC-AUC": round(roc_auc_score(y_test,y_prob),4)
    }


# -------------------------------------------------------------
# Logistic Regression Comparison
# -------------------------------------------------------------

# Original Data
lr_original = LogisticRegression(
    max_iter=1000,
    random_state=42
)

r1 = quick_eval(
    lr_original,
    X_train,
    y_train,
    X_test,
    y_test,
    "Original"
)


# Class Weight
lr_balanced = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

r2 = quick_eval(
    lr_balanced,
    X_train,
    y_train,
    X_test,
    y_test,
    "Class Weight"
)


# SMOTE
smote = SMOTE(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(
    X_train,
    y_train
)

lr_smote = LogisticRegression(
    max_iter=1000,
    random_state=42
)

r3 = quick_eval(
    lr_smote,
    X_train_sm,
    y_train_sm,
    X_test,
    y_test,
    "SMOTE"
)

imb_df = pd.DataFrame([r1,r2,r3])

imb_df.to_csv(
    f"{REP}/imbalance_comparison.csv",
    index=False
)

print(imb_df)

print(f"\nOriginal Training Samples : {len(X_train)}")
print(f"SMOTE Training Samples    : {len(X_train_sm)}")

         Method  Accuracy  Precision  Recall  F1 Score  ROC-AUC
0      Original    0.8070     0.6711  0.5348    0.5952   0.8463
1  Class Weight    0.7424     0.5096  0.7834    0.6175   0.8461
2         SMOTE    0.7402     0.5069  0.7861    0.6164   0.8465

Original Training Samples : 5634
SMOTE Training Samples    : 8278


## # PHASE 10 — BASELINE MODELS  (Task 18)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)

from sklearn.neighbors import KNeighborsClassifier

from sklearn.naive_bayes import GaussianNB

from sklearn.svm import SVC

from xgboost import XGBClassifier

from lightgbm import LGBMClassifier


MODELS = {

    "Logistic Regression":
        LogisticRegression(max_iter=1000, random_state=42),

    "Decision Tree":
        DecisionTreeClassifier(random_state=42),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ),

    "KNN":
        KNeighborsClassifier(n_neighbors=5),

    "Naive Bayes":
        GaussianNB(),

    "SVM":
        SVC(
            probability=True,
            random_state=42
        ),

    "AdaBoost":
        AdaBoostClassifier(random_state=42),

    "Gradient Boosting":
        GradientBoostingClassifier(random_state=42),

    "XGBoost":
        XGBClassifier(
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        ),

    "LightGBM":
        LGBMClassifier(
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )
}


# -------------------------------------------------------------
# Function to train and evaluate models
# -------------------------------------------------------------

def evaluate_models(X_train, y_train, dataset_name):

    results = []
    trained_models = {}

    for name, model in MODELS.items():

        model.fit(X_train, y_train)

        trained_models[name] = model

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:,1]

        results.append({
            "Dataset": dataset_name,
            "Model": name,
            "Accuracy": round(accuracy_score(y_test,y_pred),4),
            "Precision": round(precision_score(y_test,y_pred,zero_division=0),4),
            "Recall": round(recall_score(y_test,y_pred,zero_division=0),4),
            "F1 Score": round(f1_score(y_test,y_pred,zero_division=0),4),
            "ROC-AUC": round(roc_auc_score(y_test,y_prob),4)
        })

    return pd.DataFrame(results), trained_models


# -------------------------------------------------------------
# Evaluate Original Data
# -------------------------------------------------------------

baseline_original, original_models = evaluate_models(
    X_train,
    y_train,
    "Original"
)


# -------------------------------------------------------------
# Evaluate SMOTE Data
# -------------------------------------------------------------


baseline_smote, smote_models = evaluate_models(
    X_train_sm,
    y_train_sm,
    "SMOTE"
)


# -------------------------------------------------------------
# Combine Results
# -------------------------------------------------------------

baseline_df = pd.concat(
    [baseline_original, baseline_smote],
    ignore_index=True
)

baseline_df.sort_values(
    by=["Dataset", "ROC-AUC"],
    ascending=[True, False],
    inplace=True
)

baseline_df.to_csv(
    f"{REP}/baseline_model_results.csv",
    index=False
)

print("\nBaseline Model Comparison")
print(baseline_df.to_string(index=False))


Baseline Model Comparison
 Dataset               Model  Accuracy  Precision  Recall  F1 Score  ROC-AUC
Original Logistic Regression    0.8070     0.6711  0.5348    0.5952   0.8463
Original   Gradient Boosting    0.7942     0.6448  0.5000    0.5633   0.8435
Original            AdaBoost    0.8055     0.6592  0.5535    0.6017   0.8416
Original            LightGBM    0.7921     0.6278  0.5321    0.5760   0.8310
Original       Random Forest    0.7814     0.6154  0.4706    0.5333   0.8213
Original             XGBoost    0.7786     0.5963  0.5134    0.5517   0.8203
Original         Naive Bayes    0.6615     0.4316  0.8690    0.5768   0.8050
Original                 SVM    0.7892     0.6332  0.4893    0.5520   0.7916
Original                 KNN    0.7580     0.5481  0.5027    0.5244   0.7694
Original       Decision Tree    0.7381     0.5067  0.5080    0.5073   0.6644
   SMOTE Logistic Regression    0.7402     0.5069  0.7861    0.6164   0.8465
   SMOTE            AdaBoost    0.7615     0.5358

## # PHASE 11 — MODEL EVALUATION  (Task 19)

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    auc
)

def evaluate_plots(models, X_test, y_test, dataset_name):

    print(f"\nGenerating evaluation plots ({dataset_name})")

    n_models = len(models)
    n_cols = 5
    n_rows = -(-n_models // n_cols)  # ceiling division

    fig_cm, axes_cm = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes_cm = axes_cm.flatten()

    fig_roc, axes_roc = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes_roc = axes_roc.flatten()

    fig_pr, axes_pr = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes_pr = axes_pr.flatten()

    for i, (name, model) in enumerate(models.items()):

        safe = f"{dataset_name}_{name.replace(' ', '_')}"
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        # =====================================================
        # Confusion Matrix subplot
        # =====================================================
        cm = confusion_matrix(y_test, y_pred)

        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["No Churn", "Churn"],
            yticklabels=["No Churn", "Churn"],
            ax=axes_cm[i],
            cbar=False
        )
        axes_cm[i].set_title(f"{dataset_name} - {name}", fontsize=11)
        axes_cm[i].set_xlabel("Predicted")
        axes_cm[i].set_ylabel("Actual")

        # =====================================================
        # Classification Report 
        # =====================================================
        report = classification_report(
            y_test, y_pred, output_dict=True, target_names=["No Churn", "Churn"]
        )
        pd.DataFrame(report).T.to_csv(f"{REP}/classification_report_{safe}.csv")

        # =====================================================
        # ROC Curve subplot
        # =====================================================
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        axes_roc[i].plot(fpr, tpr, lw=2, label=f"AUC = {auc(fpr, tpr):.3f}")
        axes_roc[i].plot([0, 1], [0, 1], "k--")  # random guess baseline
        axes_roc[i].set_title(f"{dataset_name} - {name}", fontsize=11)
        axes_roc[i].set_xlabel("False Positive Rate")
        axes_roc[i].set_ylabel("True Positive Rate")
        axes_roc[i].legend(fontsize=8)

        # =====================================================
        # Precision-Recall Curve subplot
        # =====================================================
        precision, recall, _ = precision_recall_curve(y_test, y_prob)
        axes_pr[i].plot(recall, precision, lw=2, label=f"AP = {auc(recall, precision):.3f}")
        axes_pr[i].set_title(f"{dataset_name} - {name}", fontsize=11)
        axes_pr[i].set_xlabel("Recall")
        axes_pr[i].set_ylabel("Precision")
        axes_pr[i].legend(fontsize=8)

        print(f"Finished {dataset_name} - {name}")

    # Hide any unused subplot slots (if n_models doesn't fill the grid evenly)
    for j in range(i + 1, len(axes_cm)):
        axes_cm[j].axis("off")
        axes_roc[j].axis("off")
        axes_pr[j].axis("off")

    # ---- Save Confusion Matrix grid ----
    plt.figure(fig_cm.number)
    plt.suptitle(f"Confusion Matrices — {dataset_name}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{PLT}/confusion_matrices_{dataset_name}.png", dpi=120)
    plt.close(fig_cm)

    # ---- Save ROC grid ----
    plt.figure(fig_roc.number)
    plt.suptitle(f"ROC Curves — {dataset_name}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{PLT}/roc_curves_{dataset_name}.png", dpi=120)
    plt.close(fig_roc)

    # ---- Save PR grid ----
    plt.figure(fig_pr.number)
    plt.suptitle(f"Precision-Recall Curves — {dataset_name}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{PLT}/pr_curves_{dataset_name}.png", dpi=120)
    plt.close(fig_pr)

    print(f"Finished {dataset_name}")

In [ ]:
print("\n PHASE 11 — Model Evaluation")

evaluate_plots(
    original_models,
    X_test,
    y_test,
    "Original"
)

evaluate_plots(
    smote_models,
    X_test,
    y_test,
    "SMOTE"
)

print("\n Evaluation completed.")


 PHASE 11 — Model Evaluation

Generating evaluation plots (Original)
Finished Original - Logistic Regression
Finished Original - Decision Tree
Finished Original - Random Forest
Finished Original - KNN
Finished Original - Naive Bayes
Finished Original - SVM
Finished Original - AdaBoost
Finished Original - Gradient Boosting
Finished Original - XGBoost
Finished Original - LightGBM
Finished Original

Generating evaluation plots (SMOTE)
Finished SMOTE - Logistic Regression
Finished SMOTE - Decision Tree
Finished SMOTE - Random Forest
Finished SMOTE - KNN
Finished SMOTE - Naive Bayes
Finished SMOTE - SVM
Finished SMOTE - AdaBoost
Finished SMOTE - Gradient Boosting
Finished SMOTE - XGBoost
Finished SMOTE - LightGBM
Finished SMOTE

 Evaluation completed.


## # PHASE 12 — HYPERPARAMETER TUNING (Task 20)

In [ ]:
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Note: param_grids now use models WITHOUT pre-fitting SMOTE;
# SMOTE is applied fresh inside each CV fold via the pipeline.
param_grids = {

    'Logistic Regression': (
        LogisticRegression(max_iter=1000, random_state=42),
        {'model__C': [0.01, 0.1, 1, 10],
         'model__solver': ['lbfgs', 'liblinear']}
    ),

    'Decision Tree': (
        DecisionTreeClassifier(random_state=42),
        {'model__max_depth': [3, 5, 10, None],
         'model__min_samples_split': [2, 5]}
    ),

    'Random Forest': (
        RandomForestClassifier(random_state=42, n_jobs=-1),
        {'model__n_estimators': [50, 100],
         'model__max_depth': [5, 10, None]}
    ),

    'KNN': (
        KNeighborsClassifier(),
        {'model__n_neighbors': [3, 5, 7, 11],
         'model__weights': ['uniform', 'distance']}
    ),

    'SVM': (
        SVC(probability=True, random_state=42),
        {'model__C': [0.1, 1, 10],
         'model__kernel': ['linear', 'rbf']}
    ),

    'XGBoost': (
        XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1),
        {'model__n_estimators': [50, 100],
         'model__max_depth': [3, 5],
         'model__learning_rate': [0.05, 0.1]}
    ),

    'LightGBM': (
        LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
        {'model__n_estimators': [50, 100],
         'model__num_leaves': [20, 31],
         'model__learning_rate': [0.05, 0.1]}
    )
}

best_tuned_models = {}
tuned_results = []

for name, (model, params) in param_grids.items():

    # SMOTE is now INSIDE the pipeline, so it's refit on only the
    # training portion of each CV fold — no leakage into validation folds.
    pipe = ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('model', model)
    ])

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=params,
        cv=3,
        scoring='roc_auc',
        n_jobs=-1
    )

    # Fit on RAW (pre-SMOTE) training data — pipeline handles resampling internally
    grid.fit(X_train, y_train)

    # Extract just the fitted classifier (not the full pipeline) so downstream
    # code (predict, feature_importances_, SHAP, etc.) works unchanged
    best_tuned_models[name] = grid.best_estimator_.named_steps['model']

    # Clean up param names for readability (strip the 'model__' prefix)
    clean_params = {k.replace('model__', ''): v for k, v in grid.best_params_.items()}

    tuned_results.append({
        'Model': name,
        'Best Parameters': str(clean_params),
        'Best ROC-AUC': round(grid.best_score_, 4)
    })

    print(f"{name} | ROC-AUC = {grid.best_score_:.4f}")

tuned_df = pd.DataFrame(tuned_results)

tuned_df.to_csv(
    f"{REP}/hyperparameter_results.csv",
    index=False
)

print("\n Hyperparameter tuning completed")

Logistic Regression | ROC-AUC = 0.8472
Decision Tree | ROC-AUC = 0.8179
Random Forest | ROC-AUC = 0.8434
KNN | ROC-AUC = 0.7917
SVM | ROC-AUC = 0.8446
XGBoost | ROC-AUC = 0.8461
LightGBM | ROC-AUC = 0.8439

 Hyperparameter tuning completed


## # PHASE 13 — FEATURE IMPORTANCE (Task 21)

In [ ]:

feature_importance_models = [
    'Random Forest',
    'XGBoost',
    'LightGBM'
]

feature_tables = []

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18,7)
)

for ax, model_name in zip(axes, feature_importance_models):

    model = best_tuned_models[model_name]

    fi = pd.DataFrame({

        'Feature': X_train_sm.columns,
        'Importance': model.feature_importances_

    })

    fi['Model'] = model_name

    fi.sort_values(
        by='Importance',
        ascending=False,
        inplace=True
    )

    feature_tables.append(fi)

    top15 = fi.head(15)

    ax.barh(
        top15['Feature'][::-1],
        top15['Importance'][::-1]
    )

    ax.set_title(
        f"{model_name} Top 15 Features",
        fontweight='bold'
    )

    ax.set_xlabel('Importance')

plt.suptitle(
    'Feature Importance Comparison',
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout()

plt.savefig(
    f"{PLT}/feature_importance.png",
    dpi=150
)

plt.close()

pd.concat(
    feature_tables,
    ignore_index=True
).to_csv(
    f"{REP}/feature_importance.csv",
    index=False
)

print(" Feature importance reports saved")

 Feature importance reports saved


## # PHASE 14 — EXPLAINABLE AI

In [ ]:
import shap
from lime.lime_tabular import LimeTabularExplainer

# Use tuned LightGBM model
model = best_tuned_models["Logistic Regression"]


# Task 22 — SHAP


print("Generating SHAP Summary...")

explainer = shap.LinearExplainer(model, X_train) 
# dont use X_train_sm here, should be the original training data before SMOTE, as SHAP expects the original feature distribution

shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10,6))

shap.summary_plot(
    shap_values,
    X_test,
    show=False
)

plt.tight_layout()

plt.savefig(
    f"{PLT}/shap_summary.png",
    dpi=150,
    bbox_inches="tight"
)

plt.close()

print("shap_summary.png saved")



# Task 23 — LIME


print("Generating LIME Explanation...")

lime_explainer = LimeTabularExplainer(

    training_data=X_train.values,

    feature_names=X_train.columns.tolist(),

    class_names=["No Churn","Churn"],

    mode="classification"

)

# Predict churn probabilities
churn_prob = model.predict_proba(X_test)[:, 1]

# Customer with highest churn probability
churn_index = np.argmax(churn_prob)

exp = lime_explainer.explain_instance(
    X_test.iloc[churn_index].values,
    model.predict_proba,
    num_features=10
)

# Save interactive explanation
exp.save_to_file(f"{REP}/lime_explanation.html")

# Save image
fig = exp.as_pyplot_figure()

plt.tight_layout()

plt.savefig(f"{PLT}/lime_explanation.png", dpi=150)

plt.close()
print("lime_explanation.png saved")

Background dataset has 5634 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=5634 when initializing the masker.


Generating SHAP Summary...
shap_summary.png saved
Generating LIME Explanation...
lime_explanation.png saved


## # PHASE 15 — MODEL COMPARISON (Task 24)

In [ ]:

comparison_results = []

# Combine baseline and tuned models
all_models = {}

# Baseline models
for name, model in smote_models.items():
    all_models[f"{name} (Baseline)"] = model

# Tuned models (replace/add)
for name, model in best_tuned_models.items():
    all_models[f"{name} (Tuned)"] = model

# Evaluate every model
for name, model in all_models.items():

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    comparison_results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "F1 Score": round(f1_score(y_test, y_pred, zero_division=0), 4),
        "ROC-AUC": round(roc_auc_score(y_test, y_prob), 4)
    })

# Create comparison table
comparison_df = pd.DataFrame(comparison_results)
comparison_df.sort_values("ROC-AUC", ascending=False, inplace=True)

comparison_df.to_csv(f"{REP}/model_comparison.csv", index=False)

print(comparison_df.to_string(index=False))


# -----------------------------
# Model Comparison Plot
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left : ROC-AUC Ranking
axes[0].barh(
    comparison_df["Model"][::-1],
    comparison_df["ROC-AUC"][::-1]
)
axes[0].set_title("ROC-AUC Comparison", fontweight="bold")
axes[0].set_xlabel("ROC-AUC")

# Right : F1 Score Comparison
axes[1].barh(
    comparison_df["Model"][::-1],
    comparison_df["F1 Score"][::-1]
)
axes[1].set_title("F1 Score Comparison", fontweight="bold")
axes[1].set_xlabel("F1 Score")

plt.tight_layout()
plt.savefig(f"{PLT}/model_comparison.png", dpi=150)
plt.close()

print("\n model_comparison.csv saved")
print("model_comparison.png saved")


                         Model  Accuracy  Precision  Recall  F1 Score  ROC-AUC
Logistic Regression (Baseline)    0.7402     0.5069  0.7861    0.6164   0.8465
   Logistic Regression (Tuned)    0.7395     0.5060  0.7888    0.6165   0.8463
           AdaBoost (Baseline)    0.7615     0.5358  0.7594    0.6283   0.8423
               XGBoost (Tuned)    0.7715     0.5506  0.7567    0.6374   0.8418
              LightGBM (Tuned)    0.7842     0.5866  0.6337    0.6093   0.8415
                   SVM (Tuned)    0.6941     0.4575  0.8209    0.5876   0.8403
  Gradient Boosting (Baseline)    0.7800     0.5721  0.6791    0.6210   0.8401
         Random Forest (Tuned)    0.7466     0.5148  0.7914    0.6238   0.8386
           LightGBM (Baseline)    0.7842     0.5926  0.5989    0.5957   0.8337
      Random Forest (Baseline)    0.7743     0.5737  0.5829    0.5782   0.8224
            XGBoost (Baseline)    0.7750     0.5772  0.5695    0.5734   0.8199
         Decision Tree (Tuned)    0.7559     0.5294 

## # PHASE 17 — MODEL SAVING (Task 26)

In [ ]:
import joblib

# Best model based on ROC-AUC
best_model_name = comparison_df.iloc[0]["Model"]
base_name = best_model_name.replace(" (Baseline)", "").replace(" (Tuned)", "")

if "(Tuned)" in best_model_name:
    best_model = best_tuned_models[base_name]
else:
    best_model = smote_models[base_name]

# Save model
joblib.dump(best_model, f"{MDL}/best_model.pkl")

# Save prediction pipeline
prediction_pipeline = {
    "scaler": scaler,
    "model": best_model,
    "feature_names": X_train_sm.columns.tolist()
}

joblib.dump(prediction_pipeline, f"{MDL}/prediction_pipeline.pkl")

print(f"\nBest Model : {best_model_name}")
print(f"ROC-AUC      : {comparison_df.iloc[0]['ROC-AUC']:.4f}")
print("best_model.pkl saved")
print("prediction_pipeline.pkl saved")


Best Model : Logistic Regression (Baseline)
ROC-AUC      : 0.8465
best_model.pkl saved
prediction_pipeline.pkl saved
